In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime


In [8]:
url = "https://traficozmg.com/"

In [9]:
response = requests.get(url)
print("Status :", response.status_code)

Status : 403


In [11]:
soup = BeautifulSoup(response.text, "html.parser")
print(soup.title.text)

Just a moment...


In [13]:
data = []

articles = soup.find_all("h2")

for article in articles:
    text = article.get_text(strip=True)
    data.append({
        "timestamp": datetime.now(),
        "avenida": "ZMG",
        "velocidad": None,
        "densidad": None,
        "detenciones": None,
        "descripcion": text
    })

df_scraped = pd.DataFrame(data)
df_scraped.head()

""


In [16]:
df_scraped = df_scraped.drop_duplicates()

if "descripcion" in df_scraped.columns:
    df_scraped = df_scraped[df_scraped["descripcion"].fillna("").str.strip() != ""]

df_scraped.head()

""


In [17]:
SCRAPED_PATH = "../data/raw/scraped_traffic.csv"
df_scraped.to_csv(SCRAPED_PATH, index=False)
print("Datos scrapeados guardados")


Datos scrapeados guardados


In [20]:
def classify_density(text):
    text = str(text).lower()
    if "fuerte" in text or "congestionado" in text:
        return 3
    elif "carga" in text:
        return 2
    else:
        return 1

if "descripcion" in df_scraped.columns:
    df_scraped["densidad"] = df_scraped["descripcion"].fillna("").apply(classify_density)
else:
    df_scraped["densidad"] = pd.Series(dtype="Int64")

df_scraped.head()

,densidad


In [21]:
DATA_RAW_PATH = "../data/raw/traffic_data.csv"
df_main = pd.read_csv(DATA_RAW_PATH)
df_combined = pd.concat([df_main, df_scraped], ignore_index=True)
df_combined.to_csv(DATA_RAW_PATH, index=False)

df_combined.tail()

,timestamp,avenida,velocidad,densidad,detenciones
10,2026-03-17 22:43:09.944193,Lopez Mateos,20.0,3,2.0
11,2026-03-17 22:43:09.958288,Lopez Mateos,15.0,3,2.0
12,2026-03-17 22:43:12.252931,Lopez Mateos,25.0,3,2.0
13,2026-03-17 22:43:12.258683,Lopez Mateos,20.0,3,2.0
14,2026-03-17 22:43:12.273623,Lopez Mateos,15.0,3,2.0



Este pipeline tiene como objetivo complementar el dataset principal mediante la recolección de información textual proveniente de fuentes públicas relacionadas con el tráfico urbano.

A diferencia del primer pipeline, donde los datos son estructurados y cuantitativos, en este caso se trabaja con datos no estructurados en forma de texto. Estos datos representan descripciones humanas del estado del tráfico en tiempo real, como reportes de congestión, carga vehicular o fluidez.

Para hacer útil esta información dentro del análisis, se realiza un proceso de transformación en el cual el texto es interpretado y convertido en variables numéricas. En este caso, se define una variable de densidad basada en palabras clave presentes en las descripciones. Por ejemplo, términos como "congestión" o "tráfico pesado" se asocian a niveles altos de densidad, mientras que descripciones más neutrales se consideran como niveles bajos.

Este enfoque permite integrar información cualitativa dentro de un sistema cuantitativo, lo cual resulta especialmente útil en contextos donde los datos estructurados son limitados o incompletos.

Es importante destacar que los datos obtenidos mediante web scraping pueden presentar ruido, inconsistencias o ambigüedad, por lo que su interpretación debe realizarse con cautela. Sin embargo, su valor radica en aportar contexto adicional al análisis del tráfico, permitiendo contrastar los resultados obtenidos a partir de métricas como el Traffic Stability Index (TSI).

En conjunto, este pipeline no busca reemplazar los datos cuantitativos, sino enriquecer el análisis mediante una capa adicional de información basada en la percepción y reporte humano del tráfico.